# 01: Generating Ground Truth Data


In [2]:
import json

with open("../../data/corpus.jsonl") as f:
    documents = [json.loads(line) for line in f]

len(documents)

709

In [3]:
doc = documents[0]
print(doc["tip_id"])
print(doc["tip"])

tip-891255eed785
Take the Warding Totem (yellow trinket) as your starting trinket in the majority of matchups, since it lets you ward your lane in the early game.


In [5]:
from pydantic import BaseModel

class GeneratedQuestion(BaseModel):
    question: str

In [6]:
data_gen_instructions = """
You are a bot lane player — ADC or support — reviewing a game you just played
and trying to work out what you did wrong.

You'll be shown one coaching tip. Write a single question you might ask that this
tip answers.

Ask it from the situation, not the solution: start from what you'd have noticed
in-game — the mistake, the moment it went wrong, the thing that felt off — the
way you'd phrase it before you knew the fix. Lean away from the tip's own wording
(champion and item names, terms like "vision score" or "roam timing") and describe
the moment in your own words — but don't force it when the plain phrasing is the
natural one.

Write the way people actually ask online: not too formal, not too short, not too
long. One complete question.
""".strip()

In [14]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
anthropic_client = Anthropic()

In [15]:
user_prompt = f"Coaching tip:\n{doc['tip']}"

In [16]:
messages = [
    {"role": "user", "content": user_prompt}
]

In [17]:
response = anthropic_client.messages.parse(
    model="claude-haiku-4-5",
    system=data_gen_instructions,
    max_tokens=1000,
    messages=messages,
    output_format=GeneratedQuestion
)
print(response)

ParsedMessage[TypeVar](id='msg_011Cdy73tUmHbYR2Zw6ARfw3', container=None, content=[ParsedTextBlock[TypeVar](citations=None, text='{"question": "Should I be starting with the yellow trinket instead of my current trinket choice so I can actually ward our lane early on?"}', type='text', parsed_output=GeneratedQuestion(question='Should I be starting with the yellow trinket instead of my current trinket choice so I can actually ward our lane early on?'))], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=393, output_tokens=34, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [18]:
result = response.parsed_output
print(result)

question='Should I be starting with the yellow trinket instead of my current trinket choice so I can actually ward our lane early on?'


In [20]:
from carryia.eval.evaluation_utils import llm_structured

/Users/mattheworga/Documents/dev/Carryia/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
result, usage = llm_structured(
    anthropic_client,
    data_gen_instructions,
    user_prompt,
    GeneratedQuestion
)

print(result.question)  # type: ignore

Why do I keep getting caught out in lane when the enemy jungler shows up, and should I be buying control wards earlier?


In [22]:
from carryia.eval.evaluation_utils import calc_price

In [23]:
cost = calc_price(usage)

cost

{'input_cost': 0.000393,
 'output_cost': 0.00017499999999999997,
 'total_cost': 0.0005679999999999999}

In [24]:
records = [{
    "question": result.question,  # type: ignore
    "seed_tip_id": doc["tip_id"],
}]

records

[{'question': 'Why do I keep getting caught out in lane when the enemy jungler shows up, and should I be buying control wards earlier?',
  'seed_tip_id': 'tip-891255eed785'}]

In [25]:
from carryia.eval.evaluation_utils import llm_structured_retry

In [26]:
def generate_ground_truth(doc):
    user_prompt = f"Coaching tip:\n{doc['tip']}"

    out, usage = llm_structured_retry(
        anthropic_client,
        data_gen_instructions,
        user_prompt,
        GeneratedQuestion
    )

    records = [{
        "question": out.question,
        "seed_tip_id": doc["tip_id"],
    }]

    return records, usage

In [27]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
  records, usage = generate_ground_truth(doc)
  ground_truth.extend(records)
  usages.append(usage)

100%|██████████| 5/5 [00:07<00:00,  1.40s/it]


In [28]:
from concurrent.futures import ThreadPoolExecutor
from carryia.eval.evaluation_utils import map_progress

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 709/709 [02:51<00:00,  4.13it/s]


In [30]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

709

In [24]:
from carryia.eval.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08738249999999999

In [31]:
from carryia.eval.evaluation_utils import calc_total_price

calc_total_price(usages)

0.3986509999999996

In [32]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [34]:
df_ground_truth.to_json("../../data/ground_truth.jsonl", orient="records", lines=True)